# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) FAIR² dataset using the `mlcroissant` library.

The dataset details protein abundance, modifications, and sequences in human samples, with exports from mass spectrometry analysis. Metadata is defined in [FAIR² Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

### Dataset Source
The dataset source is provided via a Croissant schema URL. All dataset entities are referenced by their `@id` fields for transparency and reproducibility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"\nCitation: {dataset.metadata.cite_as}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.date_published}")


## 2. Data Overview
Let's review the available record sets, the fields within each, and their respective `@id`s.


In [ ]:
# List all record sets in the dataset and their fields
record_sets = dataset.record_sets
print(f"Record sets found in the dataset ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, type: {field.data_type})")
    print("")


## 3. Data Extraction
We can load data from each record set. Here, we select the primary record set containing protein quantification results for initial exploration.


In [ ]:
# For illustration, select the main protein quantification record set using its @id
# (substitute your field's @id if you wish to change record sets)

# Dynamically fetch all record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Use the first record set for demonstration (edit idx to select a different one)
primary_record_set_id = record_set_ids[0]
print(f"DataFrame columns in {primary_record_set_id} record set:\n{dataframes[primary_record_set_id].columns.tolist()}")
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Explore and process the data. For example, filter on a numeric field such as `cr:normalized_abundance`, normalize values, and group by protein description or similar fields. Adjust the field `@id` as shown in the overview above.


In [ ]:
# Define common field IDs for demonstration
# Please refer to the printout in section 2 to use correct @id values for your case
numeric_field = None
group_field = None
# Find the first float/integer field for numeric analysis
for field in dataset.record_set(primary_record_set_id).fields:
    if field.data_type in ['Float', 'Number', 'Integer']:
        numeric_field = field.id
        break

# Find the first text/categorical field for grouping
for field in dataset.record_set(primary_record_set_id).fields:
    if field.data_type in ['Text', 'String'] and field.id != numeric_field:
        group_field = field.id
        break

if numeric_field is None:
    raise Exception("No numeric field found in record set.")
else:
    print(f"Using {numeric_field} as the numeric field.")

df = dataframes[primary_record_set_id]
# Convert the numeric field to float if needed
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter:
threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.4f} (mean value):")
print(filtered_df.head())

# Normalize the numeric field for filtered records
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available
if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'std', 'count'])
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo suitable categorical grouping field found.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare grouped means if applicable.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group if available
if group_field is not None and group_field in df.columns:
    plt.figure(figsize=(12, 6))
    top_groups = df[group_field].value_counts().index[:10]
    sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
    plt.title(f'{numeric_field} grouped by {group_field} (top 10 groups)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## 6. Conclusion
This notebook demonstrated how to load, process, and visualize data from the FAIR² [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant`, referencing record sets and fields by their `@id` for reproducibility.

You can further extend this workflow for downstream statistical analysis, machine learning, or domain-specific protein feature engineering using the designated field `@id`s.